# 드럼리스 MR 개선 후보 비교 — GPU 서버 실행 (S15P31A307-165)

분리 후보 8개로 ENST 64곡과 MUSDB test 50곡(곡마다 30초)을 분리해 드럼 SDR과 드럼리스 MR 품질(fullness·bleedless·음량)을 재고, 데모 3곡을 후보마다 서비스 명령으로 분리해 귀로 비교할 mp3를 만든다. **위에서부터 차례로 실행한다.**

**코드 폴더를 따로 씀.** -164 작업이 쓰는 `ssaplash/S15P31A307`은 건드리지 않는다. -165 코드는 `ssaplash/work-165/S15P31A307`에 풀고, 파이썬 환경은 기존 `ssaplash/S15P31A307/AI/.venv`를 `PYTHONPATH`로 -165 코드를 앞세워 같이 쓴다. ENST 로더는 -164 코드에만 있어서, ENST를 MUSDB 폴더 모양으로 내보내는 4번 셀만 기존 코드 폴더로 돈다.

**준비물.** 로컬 -165 작업 폴더에서 만든 `S15P31A307-165_code.zip`을 `ssaplash/upload/`에, 이 노트북을 `ssaplash/notebooks/`에 올린다. 데모 3곡은 `ssaplash/upload/demo_songs/`에 이미 있다.

**순서.** 1 설정 → 2 코드 풀기 → 3 패키지 추가·확인 → 4 데이터 링크·ENST 내보내기(처음 한 번 5분 안팎) → 5 본측정 시작(후보 8개를 작업 8개로 따로 띄움, 백그라운드) → 6 진행 확인(여러 번, 2~3시간) → 7 데모 3곡 후보별 분리(20분 안팎) → 8 결과 묶기.

**끊기면.** 5번은 nohup이라 브라우저를 닫아도 돈다. 커널이 다시 시작됐으면 1번 다음 5번을 다시 친다. 이미 시작한 후보는 로그 파일이 있어 건너뛰고 `이미 시작함`만 찍힌다. 그다음 6번으로 이어 간다.

**추가 후보 G(10/31).** 후보 8개 측정이 끝난 뒤 TTA(극성 반전·채널 바꿈 입력을 더 분리해 평균, `bs_roformer_sw_tta.yaml`)를 후보 G로 더함. 코드 zip을 다시 올리고 1 → 2 → 5-1 → 6 → 7 → 8 순서로 치면 G만 새로 돈다. 분리 시간이 3배라 ENST 64곡 분리만 15분 안팎, 점수 계산까지 1시간 30분 안팎.

**추가 후보 I·리미터(10/31 설계 v0.4).** 드럼을 90%·95%만 뺀 MR(`bs_roformer_sw_r090`·`_r095`), 반주 리미터를 건 A(`bs_roformer_sw_limit`), 둘을 합친 것(`bs_roformer_sw_r095_limit`) 4개를 더함. 코드 zip을 다시 올리고 1 → 2 → 5 → 5-1 → 5-2 → 6 → 7 → 7-1 → 8 순서로 친다. 5·5-1은 이미 시작한 후보를 건너뛰고 `start` 함수만 정의함. G까지 5개가 같이 돌아 2시간 안팎. 7-1은 드럼 파일 저장 처리를 바꾼 뒤 데모 3곡 채보 결과가 그대로인지 봄.

**왜 나눠 돌리나.** 곡 하나 분리는 4초 안팎인데 점수 계산(SDR)이 CPU로 곡당 80초 안팎 걸린다. 후보 8개를 한 작업으로 차례로 돌리면 ENST만 11시간이 넘어서, 후보마다 작업을 따로 띄운다.

In [ ]:
# 1. 설정. 지정받은 GPU는 6번
import os
import subprocess
import sys
from pathlib import Path

GPU_ID = "6"
HOME = Path.home()
WORK = HOME / "ssaplash"
MAIN = WORK / "S15P31A307" / "AI"                  # -164 코드 폴더. .venv와 ENST 로더만 빌려 씀
W = WORK / "work-165" / "S15P31A307" / "AI"        # -165 코드 폴더
VPY = MAIN / ".venv" / "bin" / "python"
UPLOAD = WORK / "upload"
RESULTS = WORK / "results"
FFMPEG = WORK / "tools" / "ffmpeg-7.0.2-amd64-static"
ENV = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES=GPU_ID,
    PYTHONUTF8="1",
    MPLBACKEND="Agg",  # 커널의 Jupyter 전용 그래프 설정을 하위 명령이 물려받으면 멈춤
    PATH=f"{FFMPEG}:{os.environ['PATH']}",  # SDR 계산 패키지가 ffmpeg를 찾음
    PYTHONPATH=str(W / "src"),  # 기존 .venv로 -165 코드를 씀
    # 작업 8개를 같이 돌리므로 numpy가 작업마다 코어 전부를 쓰지 않게 4개로 막음 (8×4=32코어)
    OMP_NUM_THREADS="4",
    OPENBLAS_NUM_THREADS="4",
    MKL_NUM_THREADS="4",
)
RESULTS.mkdir(parents=True, exist_ok=True)
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total --format=csv
!df -h $HOME | tail -1
!ls -la $UPLOAD/S15P31A307-165_code.zip
# 확인: GPU 한 줄(H200, 남은 메모리 20GB 이상), 디스크 여유 20GB 이상, 코드 zip이 오늘 날짜

In [ ]:
# 2. -165 코드 풀기. 기존 S15P31A307 폴더와 다른 곳이라 -164 작업에 영향 없음 (여러 번 쳐도 됨)
(WORK / "work-165").mkdir(exist_ok=True)
!{sys.executable} -m zipfile -e $UPLOAD/S15P31A307-165_code.zip $WORK/work-165
!ls $W/src/ssaplash_ai/bench/metrics/mr_quality.py $W/configs/bench/models/ensemble/sw_scnet_xl_ihf.yaml $W/scripts/export_enst_as_musdb.py
# 확인: 파일 3개가 그대로 찍힘

In [ ]:
# 3. 기존 .venv에 음량 측정 패키지 하나 추가하고, -165 코드를 쓰는지 확인 (여러 번 쳐도 됨)
!{sys.executable} -m uv pip install --python $VPY -q pyloudnorm
r = subprocess.run([str(VPY), "-c", "import ssaplash_ai, pyloudnorm, torch; print(ssaplash_ai.__file__); print('cuda', torch.cuda.is_available())"], env=ENV, capture_output=True, text=True)
print(r.stdout, r.stderr[-500:])
# 확인: 첫 줄이 .../work-165/S15P31A307/AI/src/ssaplash_ai/__init__.py, 둘째 줄 'cuda True'

In [ ]:
# 4. 데이터 링크와 ENST 내보내기. ENST 내보내기는 처음 한 번만 5분 안팎, 이미 있으면 건너뜀 (여러 번 쳐도 됨)
data = W / "data"
for name, target in [
    ("models", MAIN / "data" / "models"),
    ("musdb18hq", WORK / "datasets" / "musdb" / "musdb18hq"),
    ("enst_gain1_musdb", WORK / "datasets" / "enst-gain1-musdb"),
]:
    link = data / name
    if not link.exists():
        link.symlink_to(target)
# ENST 로더가 있는 -164 코드 폴더에서 실행 (PYTHONPATH 없이)
env_main = {k: v for k, v in ENV.items() if k != "PYTHONPATH"}
r = subprocess.run([str(VPY), str(W / "scripts" / "export_enst_as_musdb.py"), "configs/bench/datasets/enst_minus_one.yaml",
                    str(WORK / "datasets" / "enst-gain1-musdb" / "test"), "--accomp-gain", "1.0"],
                   cwd=MAIN, env=env_main, capture_output=True, text=True)
print(r.stdout, r.stderr[-800:])
r = subprocess.run([str(VPY), "-m", "ssaplash_ai.bench.cli", "check-data", "configs/bench/datasets/enst_gain1_musdb.yaml", "configs/bench/datasets/musdb18hq_test.yaml"],
                   cwd=W, env=ENV, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-500:])
# 확인: '새로 쓴 곡 64개, 폴더 안 곡 64개'(두 번째부터는 0개, 64개).
#       check-data가 enst_gain1_musdb 곡 64개·누락 0건, musdb18hq_test 곡 50개·누락 0건

In [ ]:
# 5. 본측정 시작 (백그라운드). 후보마다 작업 하나씩, 8개를 동시에 띄움
#    작업 하나는 ENST 64곡 전체를 끝낸 뒤 MUSDB 50곡 30초 구간을 돎. 이미 시작한 후보는 건너뜀 (여러 번 쳐도 됨)
import shlex

CANDS = [
    "configs/bench/models/msst/bs_roformer_sw.yaml",          # A 지금 설정
    "configs/bench/models/msst/bs_roformer_sw_ov4.yaml",      # B 겹침 4
    "configs/bench/models/msst/bs_roformer_musdb.yaml",       # C 다른 모델
    "configs/bench/models/msst/scnet_xl_ihf.yaml",            # C
    "configs/bench/models/demucs/htdemucs_ft.yaml",           # C
    "configs/bench/models/ensemble/sw_scnet_xl_ihf.yaml",     # D 드럼 평균
    "configs/bench/models/msst/bs_roformer_sw_highret.yaml",  # E 고음 지속음 되돌리기
    "configs/bench/models/msst/bs_roformer_sw_sum.yaml",      # F 다섯 스템 합치기
]


def run_ids(cand: str) -> tuple[str, str]:
    stem = Path(cand).stem
    # A는 처음 한 작업으로 돌던 mrq-enst를 그대로 써서 끝난 곡을 건너뜀
    enst = "mrq-enst" if stem == "bs_roformer_sw" else f"mrq-enst-{stem}"
    return enst, f"mrq-musdb50-{stem}"


def start(cands: list[str]) -> None:
    """후보마다 ENST → MUSDB 순서로 도는 작업 하나를 nohup으로 띄움. 로그 파일이 있으면 건너뜀."""
    for cand in cands:
        stem = Path(cand).stem
        log = W / f"mrq-{stem}.log"
        if log.exists():
            print("이미 시작함", stem)
            continue
        enst, musdb = run_ids(cand)
        base = [str(VPY), "-m", "ssaplash_ai.bench.cli", "run", "--models", cand, "--out", "data/results", "--mlflow-uri", "none"]
        cmds = [
            base + ["--dataset", "configs/bench/datasets/enst_gain1_musdb.yaml", "--run-id", enst],
            base + ["--dataset", "configs/bench/datasets/musdb18hq_test.yaml", "--clip", "30", "60", "--run-id", musdb],
        ]
        script = " ; ".join(shlex.join(c) for c in cmds)
        p = subprocess.Popen(["nohup", "bash", "-c", script], cwd=W, env=ENV, stdout=open(log, "w"),
                             stderr=subprocess.STDOUT, start_new_session=True)
        print("시작", stem, "pid", p.pid)


start(CANDS)
# 확인: 처음엔 '시작 bs_roformer_sw pid 숫자'부터 8줄. 다시 치면 '이미 시작함 ...' 8줄. 바로 6번으로


In [ ]:
# 5-1. 추가 후보 G(TTA) 측정 시작 (백그라운드, 10/31 추가). 5번과 같은 방식, 이미 시작했으면 건너뜀 (여러 번 쳐도 됨)
CANDS_EXTRA = [
    "configs/bench/models/msst/bs_roformer_sw_tta.yaml",      # G 극성 반전·채널 바꿈 평균(TTA)
]
start(CANDS_EXTRA)
# 확인: '시작 bs_roformer_sw_tta pid 숫자' 한 줄. 다시 치면 '이미 시작함 bs_roformer_sw_tta'. 바로 6번으로


In [ ]:
# 5-2. 추가 후보 I(드럼 일부만 빼기)·리미터 측정 시작 (백그라운드, 설계 v0.4). 이미 시작했으면 건너뜀 (여러 번 쳐도 됨)
#      네 후보 모두 드럼 트랙은 A와 같고 반주만 다름. 드럼 SDR은 A와 같게 나와야 정상
CANDS_V04 = [
    "configs/bench/models/msst/bs_roformer_sw_r090.yaml",        # I 드럼 90%만 뺌
    "configs/bench/models/msst/bs_roformer_sw_r095.yaml",        # I 드럼 95%만 뺌
    "configs/bench/models/msst/bs_roformer_sw_limit.yaml",       # A + 반주 리미터
    "configs/bench/models/msst/bs_roformer_sw_r095_limit.yaml",  # I 95% + 반주 리미터
]
ALL_CANDS = CANDS + CANDS_EXTRA + CANDS_V04
start(CANDS_V04)
# 확인: '시작 bs_roformer_sw_r090 pid 숫자'부터 4줄. 다시 치면 '이미 시작함 ...' 4줄. 바로 6번으로


In [ ]:
# 6. 진행 확인 (여러 번 쳐도 됨). 후보마다 ENST 64행, MUSDB 50행이 되면 끝
for cand in ALL_CANDS:
    stem = Path(cand).stem
    counts = []
    for rid in run_ids(cand):
        csv_path = W / "data" / "results" / rid / "results.csv"
        lines = csv_path.read_text(encoding="utf-8").splitlines()[1:] if csv_path.exists() else []
        counts.append(f"끝난 곡 {len(lines)}개 중 실패 {sum('Error' in line for line in lines)}개")
    alive = subprocess.run(["pgrep", "-f", f"models {cand} "], capture_output=True).returncode == 0
    last = (W / f"mrq-{stem}.log").read_text(encoding="utf-8", errors="replace").strip().splitlines()[-1:]
    print(f"{stem}: ENST {counts[0]} / MUSDB {counts[1]} / {'도는 중' if alive else '끝났거나 안 돌고 있음'}")
    for line in last:
        print("  ", line[:150])
print("load", os.getloadavg())
# 확인: 후보마다 행 수가 늘고 로그 끝이 '[모델이름] 곡이름: sdrDrums=...'. 다 끝나면 13줄 모두 'ENST 끝난 곡 64개 중 실패 0개 / MUSDB 끝난 곡 50개 중 실패 0개 / 끝났거나 안 돌고 있음'
#       '끝났거나 안 돌고 있음'인데 행이 모자라면 그 후보 로그 끝 오류를 적어 둠


In [ ]:
# 7. 데모 3곡을 후보마다 서비스 명령으로 분리하고 들어 볼 mp3를 만듦. 이미 한 곡은 건너뜀 (여러 번 쳐도 됨)
DEMO = RESULTS / "mr_quality_demo"
songs = sorted(p for p in (UPLOAD / "demo_songs").iterdir() if p.suffix.lower() in {".mp3", ".wav", ".flac", ".opus"})
for cand in ALL_CANDS:
    name = Path(cand).stem
    sep_yaml = W / "data" / f"separation_{name}.yaml"
    sep_yaml.write_text(f"model_config: {cand}\ndevice: cuda\noutput:\n  sample_rate: 44100\n  format: flac\nmax_duration_sec: 600\n", encoding="utf-8")
    for song in songs:
        out = DEMO / name / song.stem
        if not (out / "separation.json").exists():
            r = subprocess.run([str(VPY), "-m", "ssaplash_ai.chart.cli", "separate", str(song), "--out", str(out), "--config", str(sep_yaml)],
                               cwd=W, env=ENV, capture_output=True, text=True)
            print(name, song.name, "종료 코드", r.returncode, (r.stderr.strip() or "")[-300:])
        for stem in ("accompaniment", "drums"):
            mp3 = out / f"{stem}.mp3"
            if (out / f"{stem}.flac").exists() and not mp3.exists():
                subprocess.run([str(FFMPEG / "ffmpeg"), "-loglevel", "error", "-y", "-i", str(out / f"{stem}.flac"), "-b:a", "320k", str(mp3)], check=True)
!ls $RESULTS/mr_quality_demo | wc -l; find $RESULTS/mr_quality_demo -name "*.mp3" | wc -l
# 확인: 곡마다 '종료 코드 0'(처음 실행 때만 찍힘). 마지막 두 줄이 13(후보 수)과 78(후보 13 × 곡 3 × 2개)

In [ ]:
# 7-1. 데모 3곡 채보 타격 수 비교. 바꾸기 전(A, 두 파일 공통 이득으로 저장한 드럼)과 바꾼 뒤(I 95% + 리미터,
#      드럼은 자기 피크 이득만)의 드럼 파일을 서비스 채보 명령으로 채보해 패드별 타격 수를 셈 (여러 번 쳐도 됨)
import csv
import json
from collections import Counter

PAIRS = {"before": "bs_roformer_sw", "after": "bs_roformer_sw_r095_limit"}
rows = []
for song in songs:
    counts = {}
    for label, name in PAIRS.items():
        src_dir = DEMO / name / song.stem
        out = src_dir / "transcribe"
        if not (out / "chart.json").exists():
            r = subprocess.run([str(VPY), "-m", "ssaplash_ai.chart.cli", "transcribe", str(src_dir / "drums.flac"), "--out", str(out)],
                               cwd=W, env=ENV, capture_output=True, text=True)
            print(label, song.name, "종료 코드", r.returncode, (r.stderr.strip() or "")[-300:])
        chart = json.loads((out / "chart.json").read_text(encoding="utf-8"))
        counts[label] = Counter(n["pad"] for n in chart["charts"][0]["notes"])
        meta = json.loads((src_dir / "separation.json").read_text(encoding="utf-8"))
        gain = meta.get("drums_gain_db", meta.get("output_gain_db"))
        print(f"{song.stem} {label}: 드럼 이득 {gain} dB, 타격 {sum(counts[label].values())}개")
    for pad in sorted(set(counts["before"]) | set(counts["after"])):
        rows.append({"song": song.stem, "pad": pad, "before": counts["before"][pad], "after": counts["after"][pad],
                     "diff": counts["after"][pad] - counts["before"][pad]})
with open(DEMO / "transcribe_compare.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["song", "pad", "before", "after", "diff"])
    w.writeheader()
    w.writerows(rows)
for row in rows:
    print(row)
# 확인: 곡마다 before는 드럼 이득이 -3~-4dB대, after는 0에 가깝거나 더 작은 음수. 패드별 diff가 0이거나 곡당 몇 개 안쪽
#       diff가 한 패드에서 10개를 넘으면 그 곡·패드를 적어 둠 (드럼 음량이 바뀌어 검출이 달라진 것)


In [ ]:
# 8. 결과 묶기. 설계 v0.4에서 새로 돈 후보 5개 결과와 데모 파일만 묶음. 결과 폴더째 넣어야 폴더 이름이 남음
#    (본측정이 끝난 뒤, 여러 번 쳐도 됨). results 폴더의 mr_quality_v04.zip 을 내려받음
NEW = [Path(c).stem for c in CANDS_EXTRA + CANDS_V04]
dirs = " ".join(f"mrq-enst-{s} mrq-musdb50-{s}" for s in NEW)
demo_dirs = " ".join(f"mr_quality_demo/{s}" for s in NEW)
!cd $W/data/results && rm -f $RESULTS/mr_quality_v04.zip && {sys.executable} -m zipfile -c $RESULTS/mr_quality_v04.zip {dirs}
!cd $RESULTS && find {demo_dirs} mr_quality_demo/bs_roformer_sw mr_quality_demo/transcribe_compare.csv \( -name "*.mp3" -o -name "*.json" -o -name "*.csv" \) > /tmp/mrq_demo_list.txt && {sys.executable} -c "import zipfile;z=zipfile.ZipFile('mr_quality_v04.zip','a');[z.write(p.strip()) for p in open('/tmp/mrq_demo_list.txt')]"
!ls -la $RESULTS/mr_quality_v04.zip
!md5sum $RESULTS/mr_quality_v04.zip
!{sys.executable} -m zipfile -l $RESULTS/mr_quality_v04.zip | grep -c results.csv
# 확인: 'Duplicate name' 경고 없음, 200MB 안팎 zip 하나와 md5 값, results.csv 10개


## 9. 팀 MLflow 기록

GPU 서버는 팀 MLflow 서버에 닿지 않아 위 측정은 `--mlflow-uri none`으로 돌렸음. 결과 폴더(`mr_quality.zip`, `mr_quality_v04.zip`)를 로컬 `AI/data/results/`에 풀어 두고, -164 브랜치의 `ssaplash-bench relog`가 develop-AI에 들어온 뒤 11/1 이후 다른 결과와 한꺼번에 올림.
